In [4]:
import os
import subprocess
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score

# ==========================================
# 1. CONFIGURARE (MODIFICI AICI DOAR CE E CAZUL)
# ==========================================
SEED = 42
REPO_OWNER_AND_NAME = "RapiteanuCarol/polero-nitro-ai"  # ex: "andreigeorgeduta/polero-hackathon"
USE_PRIVATE_REPO = True
KAGGLE_SECRET_TOKEN_NAME = "GITHUB_TOKEN"     # adaugat in Kaggle -> Add-ons -> Secrets

# Etapa 1 (rapid): "dumitrescustefan/bert-base-romanian-cased-v1"
# Etapa 2 (greu):  "BAAI/bge-m3" sau "xlm-roberta-large"
MODEL_NAME = "dumitrescustefan/bert-base-romanian-cased-v1"
MAX_LEN = 512
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
WEIGHT_DECAY = 0.01

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

# ==========================================
# 2. CLONARE REPO DE PE GITHUB
# ==========================================
repo_dir = "repo_polero"
if not os.path.exists(repo_dir):
    if USE_PRIVATE_REPO:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret(KAGGLE_SECRET_TOKEN_NAME)
            clone_url = f"https://{token}@github.com/{REPO_OWNER_AND_NAME}.git"
        except Exception as e:
            raise RuntimeError(
                "Nu am putut citi token-ul din Kaggle Secrets. "
                "Adauga secretul 'GITHUB_TOKEN' din meniul Add-ons -> Secrets sau seteaza USE_PRIVATE_REPO = False."
            ) from e
    else:
        clone_url = f"https://github.com/{REPO_OWNER_AND_NAME}.git"

    subprocess.run(["git", "clone", clone_url, repo_dir], check=True)

print("Fisiere in repo:", os.listdir(repo_dir))

# ==========================================
# 3. INCARCARE DATE & FOLDS
# ==========================================
data_dir = os.path.join(repo_dir, "data")
if os.path.exists(data_dir):
    print("Fisiere in data/:", os.listdir(data_dir))

def find_file(candidates, folder):
    for root, _, files in os.walk(folder):
        for name in candidates:
            if name in files:
                return os.path.join(root, name)
    raise FileNotFoundError(f"Nu am gasit niciunul din fisierele {candidates} in {folder}")

train_path = find_file(
    ["train_context.csv", "train_preprocessed.csv", "train_folds.csv", "train_data.csv", "train.csv"],
    repo_dir
)
test_path = find_file(
    ["test_context.csv", "test_preprocessed.csv", "test_data.csv", "test.csv"],
    repo_dir
)
folds_path = find_file(
    ["folds.csv", "train_folds.csv", "train_context.csv", "train_preprocessed.csv"],
    repo_dir
)

print("Am gasit:\n - train:", train_path, "\n - test:", test_path, "\n - folds:", folds_path)

train_df = pd.read_csv(train_path)
test_df  = pd.read_csv(test_path)
folds_df = pd.read_csv(folds_path)

# Aliniem fold-ul pe datapointID daca exista coloana, altfel pe index
if "fold" in train_df.columns:
    pass
elif "fold" in folds_df.columns and "datapointID" in folds_df.columns and "datapointID" in train_df.columns:
    train_df = train_df.merge(folds_df[["datapointID", "fold"]], on="datapointID", how="left")
elif "fold" in folds_df.columns and len(folds_df) == len(train_df):
    train_df["fold"] = folds_df["fold"].values
else:
    raise ValueError("Verifica structura fisierului de folds (astept coloana 'fold').")

assert train_df["fold"].isna().sum() == 0, "Exista randuri fara fold!"
print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)

# Definim ordinea strict alfabetica a celor 9 clase din Subtask 2
EVASION_CLASSES = sorted(train_df["evasion_label"].unique().tolist())
assert len(EVASION_CLASSES) == 9, f"Asteptam 9 clase, am gasit {len(EVASION_CLASSES)}: {EVASION_CLASSES}"
label2id = {c: i for i, c in enumerate(EVASION_CLASSES)}
id2label = {i: c for i, c in enumerate(EVASION_CLASSES)}
train_df["label"] = train_df["evasion_label"].map(label2id)
assert train_df["label"].isna().sum() == 0, "Exista etichete nemapate!"

# Mapare deterministica Subtask 2 (9 clase) -> Subtask 1 (3 clase)
EVASION_TO_CLARITY = {
    "Explicit": "Clear Reply",
    "Implicit": "Ambivalent",
    "General": "Ambivalent",
    "Partial/half-answer": "Ambivalent",
    "Dodging": "Ambivalent",
    "Deflection": "Ambivalent",
    "Declining to answer": "Clear Non-Reply",
    "Claims ignorance": "Clear Non-Reply",
    "Clarification": "Clear Non-Reply",
}

# ==========================================
# 4. CONSTRUIRE TEXT CU CONTEXT (TURN_GAP == 1)
# ==========================================
def attach_immediate_context_if_missing(df_tr, df_te):
    # Daca Bianca a pregatit deja coloanele, le folosim direct; altfel le calculam aici
    if "turn_gap" in df_tr.columns and "turn_gap" in df_te.columns:
        return df_tr, df_te

    combined = pd.concat(
        [df_tr.assign(_split="train"), df_te.assign(_split="test")],
        ignore_index=True
    )
    combined["_orig_idx"] = np.arange(len(combined))
    combined = combined.sort_values(["url", "question_order"], kind="mergesort").reset_index(drop=True)

    same_url = combined["url"] == combined["url"].shift(1)
    combined["turn_gap"] = np.where(same_url, combined["question_order"] - combined["question_order"].shift(1), np.nan)
    combined["prev_q"] = np.where(same_url & (combined["turn_gap"] == 1), combined["interview_question"].shift(1), "")
    combined["prev_a"] = np.where(same_url & (combined["turn_gap"] == 1), combined["interview_answer"].shift(1), "")

    combined = combined.sort_values("_orig_idx").reset_index(drop=True)
    tr_out = combined[combined["_split"] == "train"].drop(columns=["_split", "_orig_idx"]).reset_index(drop=True)
    te_out = combined[combined["_split"] == "test"].drop(columns=["_split", "_orig_idx"]).reset_index(drop=True)
    return tr_out, te_out

train_df, test_df = attach_immediate_context_if_missing(train_df, test_df)

def build_input_text(row):
    pres = str(row.get("president", "")).strip()
    ctx = str(row.get("context", "")).strip()
    q = str(row.get("interview_question", "")).strip()
    a = str(row.get("interview_answer", "")).strip()

    gap = row.get("turn_gap", np.nan)
    prev_q = row.get("prev_q", row.get("prev_question", ""))
    prev_a = row.get("prev_a", row.get("prev_answer", ""))

    parts = [f"Presedinte: {pres} | Context: {ctx}"]
    if pd.notna(gap) and float(gap) == 1.0 and pd.notna(prev_q) and str(prev_q).strip() != "":
        pq = str(prev_q).strip()[:300]
        pa = str(prev_a).strip()[:300] if pd.notna(prev_a) else ""
        parts.append(f"Schimb anterior -> Intrebare: {pq} | Raspuns: {pa}")

    parts.append(f"Intrebare: {q}")
    parts.append(f"Raspuns: {a}")
    return "\n".join(parts)

train_texts = [build_input_text(r) for _, r in train_df.iterrows()]
test_texts  = [build_input_text(r) for _, r in test_df.iterrows()]
print("Exemplu input:\n", train_texts[0][:400])

# ==========================================
# 5. DATASET & DATALOADER
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class QADataset(Dataset):
    def __init__(self, texts, labels=None, max_len=512):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Ponderi inverse usor atenuate pentru clasele dezechilibrate
class_counts = np.bincount(train_df["label"].values, minlength=9)
weights = (len(train_df) / (9.0 * np.maximum(class_counts, 1))) ** 0.5
weights = weights / weights.mean()
class_weights_tensor = torch.tensor(weights, dtype=torch.float32, device=device)

def predict_probs(model, loader):
    model.eval()
    probs_list = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits.float(), dim=-1)
            probs_list.append(probs.cpu().numpy())
    return np.vstack(probs_list)

# ==========================================
# 6. ANTRENARE 5-FOLD
# ==========================================
oof_probs = np.zeros((len(train_df), 9), dtype=np.float32)
test_probs_folds = np.zeros((5, len(test_df), 9), dtype=np.float32)

test_dataset = QADataset(test_texts, labels=None, max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

for fold in range(5):
    print(f"\n========== FOLD {fold} ==========")
    tr_idx = np.where(train_df["fold"].values != fold)[0]
    va_idx = np.where(train_df["fold"].values == fold)[0]

    tr_ds = QADataset([train_texts[i] for i in tr_idx], train_df["label"].values[tr_idx], max_len=MAX_LEN)
    va_ds = QADataset([train_texts[i] for i in va_idx], train_df["label"].values[va_idx], max_len=MAX_LEN)

    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=9).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(tr_loader) * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_f1 = -1.0
    best_va_probs = None
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for batch in tr_loader:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            total_loss += loss.item()

        va_probs = predict_probs(model, va_loader)
        va_preds = va_probs.argmax(axis=1)
        macro_f1 = f1_score(train_df["label"].values[va_idx], va_preds, average="macro")
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {total_loss/len(tr_loader):.4f} | Val Macro-F1 (9 cls): {macro_f1:.4f}")

        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_va_probs = va_probs
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    oof_probs[va_idx] = best_va_probs
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    test_probs_folds[fold] = predict_probs(model, test_loader)
    print(f"Best Val Macro-F1 Fold {fold}: {best_f1:.4f}")

    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()

test_probs = test_probs_folds.mean(axis=0)

# Scor global OOF pe 9 clase si pe 3 clase (prin mapare in sus)
oof_pred_9 = oof_probs.argmax(axis=1)
oof_f1_9 = f1_score(train_df["label"].values, oof_pred_9, average="macro")

oof_pred_9_labels = [id2label[i] for i in oof_pred_9]
oof_pred_3_labels = [EVASION_TO_CLARITY[lbl] for lbl in oof_pred_9_labels]
oof_f1_3 = f1_score(train_df["clarity_label"].values, oof_pred_3_labels, average="macro")

print(f"\n>>> OOF Macro-F1 Subtask 2 (9 clase): {oof_f1_9:.4f}")
print(f">>> OOF Macro-F1 Subtask 1 (3 clase, upward mapping): {oof_f1_3:.4f}")

# ==========================================
# 7. SALVARE .NPY SI SUBMISIE (592 RANDURI)
# ==========================================
np.save("oof_encoder.npy", oof_probs)
np.save("test_encoder.npy", test_probs)

test_pred_9_idx = test_probs.argmax(axis=1)
test_pred_9_labels = [id2label[i] for i in test_pred_9_idx]
test_pred_3_labels = [EVASION_TO_CLARITY[lbl] for lbl in test_pred_9_labels]

sub_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": test_df["datapointID"].values,
    "answer": test_pred_3_labels
})
sub_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": test_df["datapointID"].values,
    "answer": test_pred_9_labels
})
sub_df = pd.concat([sub_1, sub_2], ignore_index=True)

# Cautam sample_output.csv oriunde in repo (ex: data/raw/sample_output.csv)
sample_path = None
for root, _, files in os.walk(repo_dir):
    if "sample_output.csv" in files:
        sample_path = os.path.join(root, "sample_output.csv")
        break

if sample_path is not None:
    sample_df = pd.read_csv(sample_path)
    sub_df = sample_df[["subtaskID", "datapointID"]].merge(sub_df, on=["subtaskID", "datapointID"], how="left")

assert len(sub_df) == 592, f"Asteptam 592 randuri, avem {len(sub_df)}"
assert sub_df["answer"].isna().sum() == 0, "Exista valori lipsa in answer!"

sub_df.to_csv("submission_duti_encoder.csv", index=False)

# Copiem automat si in structura de foldere a echipei (predictions/ si submissions/)
os.makedirs(os.path.join(repo_dir, "predictions"), exist_ok=True)
os.makedirs(os.path.join(repo_dir, "submissions"), exist_ok=True)
np.save(os.path.join(repo_dir, "predictions", "oof_encoder.npy"), oof_probs)
np.save(os.path.join(repo_dir, "predictions", "test_encoder.npy"), test_probs)
sub_df.to_csv(os.path.join(repo_dir, "submissions", "submission_duti_encoder.csv"), index=False)

print("Salvate cu succes in working dir si in repo_polero/predictions/ + submissions/!")

Device: cuda
Fisiere in repo: ['submissions', 'notebooks', 'README.md', 'predictions', 'data', '.gitignore', '.git']
Fisiere in data/: ['processed', 'raw']
Am gasit:
 - train: repo_polero/data/processed/train_context.csv 
 - test: repo_polero/data/processed/test_context.csv 
 - folds: repo_polero/data/processed/folds.csv
Train shape: (3278, 15) | Test shape: (296, 12)
Exemplu input:
 Presedinte: Ion Iliescu | Context: Press Conference
Intrebare: Pentru domnul Președinte Iliescu. România va fi pro sau nu Basarabia, dacă se vor pune anumite interdicții în ceea ce privește Basarabia?
Raspuns: Interdicții din partea cui?

========== FOLD 0 ==========


model.safetensors: reconstructing file:   0%|          |  0.00B /  500MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Epoch 1/3 | Train Loss: 1.9924 | Val Macro-F1 (9 cls): 0.1235
Epoch 2/3 | Train Loss: 1.7307 | Val Macro-F1 (9 cls): 0.2996
Epoch 3/3 | Train Loss: 1.4756 | Val Macro-F1 (9 cls): 0.3293
Best Val Macro-F1 Fold 0: 0.3293

========== FOLD 1 ==========


... (8 outputs left out)

Epoch 1/3 | Train Loss: 1.9637 | Val Macro-F1 (9 cls): 0.1481
Epoch 2/3 | Train Loss: 1.6681 | Val Macro-F1 (9 cls): 0.2662
Epoch 3/3 | Train Loss: 1.4157 | Val Macro-F1 (9 cls): 0.3236
Best Val Macro-F1 Fold 3: 0.3236

========== FOLD 4 ==========


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream t

Epoch 1/3 | Train Loss: 1.9931 | Val Macro-F1 (9 cls): 0.1492
Epoch 2/3 | Train Loss: 1.6982 | Val Macro-F1 (9 cls): 0.2409
Epoch 3/3 | Train Loss: 1.4556 | Val Macro-F1 (9 cls): 0.2660
Best Val Macro-F1 Fold 4: 0.2660

>>> OOF Macro-F1 Subtask 2 (9 clase): 0.3143
>>> OOF Macro-F1 Subtask 1 (3 clase, upward mapping): 0.5674
Salvate cu succes in working dir si in repo_polero/predictions/ + submissions/!


In [5]:
import os
import shutil
import subprocess
import getpass
from sklearn.metrics import classification_report

# 1. Vedem scorul detaliat pe fiecare dintre cele 9 clase
print(classification_report(
    train_df["label"].values,
    oof_probs.argmax(axis=1),
    target_names=EVASION_CLASSES,
    digits=4
))

# 2. Redenumim cu numele modelului actual (bert_ro) ca sa nu le suprascriem la pasul urmator
os.makedirs(os.path.join(repo_dir, "predictions"), exist_ok=True)
os.makedirs(os.path.join(repo_dir, "submissions"), exist_ok=True)
np.save(os.path.join(repo_dir, "predictions", "oof_bert_ro.npy"), oof_probs)
np.save(os.path.join(repo_dir, "predictions", "test_bert_ro.npy"), test_probs)
shutil.copy("submission_duti_encoder.csv", os.path.join(repo_dir, "submissions", "submission_bert_ro.csv"))

# 3. Trimitem pe GitHub
token = getpass.getpass("Token GitHub (ghp_...): ").strip()
repo_slug = REPO_OWNER_AND_NAME

subprocess.run(["git", "-C", repo_dir, "config", "user.name", "Duti"], check=True)
subprocess.run(["git", "-C", repo_dir, "config", "user.email", "duti@example.com"], check=True)
subprocess.run(["git", "-C", repo_dir, "remote", "set-url", "origin", f"https://{token}@github.com/{repo_slug}.git"], check=True)
subprocess.run(["git", "-C", repo_dir, "pull", "--rebase"], check=False)
subprocess.run(["git", "-C", repo_dir, "add", "predictions/", "submissions/"], check=True)
subprocess.run(["git", "-C", repo_dir, "commit", "-m", "Add bert-base-romanian OOF and test predictions"], check=True)
subprocess.run(["git", "-C", repo_dir, "push"], check=True)
print("Gata! Fisierele sunt pe GitHub.")

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


                     precision    recall  f1-score   support

   Claims ignorance     0.4259    0.3966    0.4107        58
      Clarification     0.5200    0.7222    0.6047        36
Declining to answer     0.3249    0.6081    0.4235       148
         Deflection     0.1636    0.0503    0.0769       179
            Dodging     0.2585    0.2657    0.2621       286
           Explicit     0.6840    0.8090    0.7413      1801
            General     0.2268    0.1621    0.1891       438
           Implicit     0.2190    0.0833    0.1207       276
Partial/half-answer     0.0000    0.0000    0.0000        56

           accuracy                         0.5415      3278
          macro avg     0.3137    0.3441    0.3143      3278
       weighted avg     0.4840    0.5415    0.5028      3278



Token GitHub (ghp_...):  ········


From https://github.com/RapiteanuCarol/polero-nitro-ai
   50630f8..74b7dc0  main       -> origin/main


Updating 50630f8..74b7dc0
Fast-forward
 notebooks/01-bianca-prep-baseline.ipynb | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 notebooks/01-bianca-prep-baseline.ipynb
[main 5e84b81] Add bert-base-romanian OOF and test predictions
 6 files changed, 1186 insertions(+)
 create mode 100644 predictions/oof_bert_ro.npy
 create mode 100644 predictions/oof_encoder.npy
 create mode 100644 predictions/test_bert_ro.npy
 create mode 100644 predictions/test_encoder.npy
 create mode 100644 submissions/submission_bert_ro.csv
 create mode 100644 submissions/submission_duti_encoder.csv
Gata! Fisierele sunt pe GitHub.


To https://github.com/RapiteanuCarol/polero-nitro-ai.git
   74b7dc0..5e84b81  main -> main
